# Kaggle runner: final C3 seeds and the test run

Same scripts as the Colab notebooks. One-time setup:
1. Datasets > New Dataset: upload `spider_bundle.zip` (private). Kaggle may unpack it; both work.
2. Notebook settings: Accelerator **GPU T4 x2** (only GPU 0 is used, so the effective batch stays 16),
   Internet **on** (needs a verified phone number). Add Input > your dataset.
3. Set `SIZE` and the queue cell, then **Save Version > Save & Run All** (background, up to 12 h).
   Download `outputs/` from the version's Output tab when it finishes.
4. For a later version that needs earlier adapters (the test run), Add Input > Your Work > this
   notebook's earlier version; `PREVIOUS` below finds its adapters.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4, same effective batch as Colab
!rm -rf /tmp/repo && git clone -q https://github.com/frangarciaproff/spider-sql-lora-ladder.git /tmp/repo
%cd /tmp/repo
!git log -1 --format="%H %s"
!pip install -q -e . peft
!pip install -q unsloth
!python --version

In [ ]:
import glob

BUNDLE_SHA256 = "aacfdd35b1a6afc369ef028cff59c39ac8639fd813c3b092fb7e5a1615e2b44f"
zips = glob.glob("/kaggle/input/**/spider_bundle.zip", recursive=True)
manifests = glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True)
if zips:
    !python scripts/get_data.py --zip {zips[0]} --dest /tmp/data --sha256 {BUNDLE_SHA256}
    DATA = "/tmp/data"
elif manifests:
    DATA = os.path.dirname(manifests[0])
    !python scripts/get_data.py --extracted {DATA}
else:
    raise FileNotFoundError("add the spider bundle dataset as an input")
os.environ["SPIDER_DIR"] = f"{DATA}/spider"
os.environ["TESTSUITE_DIR"] = f"{DATA}/testsuite"

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
print(torch.cuda.get_device_name(0), torch.cuda.device_count(), "visible")

SIZE = "0.5B"  # "0.5B" or "1.5B"
PROFILES = {  # model, tag, per-device batch, accumulation (effective 16), generation batch
    "0.5B": ("Qwen/Qwen2.5-Coder-0.5B-Instruct", "qwen05b", 8, 2, 32),
    "1.5B": ("Qwen/Qwen2.5-Coder-1.5B-Instruct", "qwen15b", 4, 4, 16),
}
MODEL, TAG, PER_DEVICE, ACCUM, GEN_BATCH = PROFILES[SIZE]
LR, EPOCHS = 2e-4, 2  # frozen 2026-10-18 (SPEC 6)
BACKEND = "unsloth"
OUT = "/kaggle/working/outputs"
ADAPTERS, PRED, RES = f"{OUT}/adapters", f"{OUT}/predictions", f"{OUT}/results"
PREVIOUS = sorted(glob.glob("/kaggle/input/**/outputs/adapters", recursive=True))
print(MODEL, TAG, "previous adapter folders:", PREVIOUS)

In [ ]:
def check(step):
    if _exit_code != 0:  # noqa: F821  (set by IPython after each ! command)
        raise RuntimeError(f"{step} failed; see the output above")


def score(run, split, condition, adapter=None, final=False):
    extra = (f"--adapter {adapter}" if adapter else "") + (" --final" if final else "")
    !python scripts/generate.py --model {MODEL} --condition {condition} --split {split} --batch-size {GEN_BATCH} --out-dir {PRED}/{run} {extra}
    check(f"generation {run} {split}")
    !python scripts/score_predictions.py --pred-dir {PRED}/{run} --split {split} --out-dir {RES}/{run}
    check(f"scoring {run} {split}")


def run_seed(seed):
    run = f"{TAG}_c3_lr{LR:g}_ep{EPOCHS}_seed{seed}"
    !python scripts/train.py --model {MODEL} --backend {BACKEND} --per-device-batch {PER_DEVICE} --grad-accum {ACCUM} --lr {LR} --epochs {EPOCHS} --seed {seed} --out-dir {ADAPTERS}/{run}
    check(f"training {run}")
    !rm -rf {ADAPTERS}/{run}/checkpoint-*
    for split in ["validation", "dev"]:
        score(run, split, "adapter", adapter=f"{ADAPTERS}/{run}/adapter")


def find_adapter(run):
    for root in [ADAPTERS, *PREVIOUS]:
        path = f"{root}/{run}/adapter"
        if os.path.isdir(path):
            return path
    raise FileNotFoundError(f"adapter for {run} not found; add the earlier version as input")


def run_test(seeds=(0, 1, 2)):
    """Review point 3 passed only. One test run per condition (SPEC 3.2)."""
    score(f"{TAG}_c1_zero_shot", "test", "zero_shot", final=True)
    score(f"{TAG}_c2_three_shot", "test", "three_shot", final=True)
    for seed in seeds:
        run = f"{TAG}_c3_lr{LR:g}_ep{EPOCHS}_seed{seed}"
        score(run, "test", "adapter", adapter=find_adapter(run), final=True)

## Queue for this version
Edit only this cell between versions (and `SIZE` above).
- Version 1: SIZE "0.5B", seeds 0, 1, 2 (about 3.5 h).
- Versions 2-3: SIZE "1.5B", one or two seeds per version (about 3 h per seed; stay under 12 h).
- Last versions, after review point 3: `run_test()` once per SIZE, with earlier versions added as input.

In [ ]:
for seed in [0, 1, 2]:
    run_seed(seed)

In [ ]:
!find {OUT} -name "*.json" | sort